# 台車モデルへ$\log$ バリア関数による制約を適用したC/GMRES

## 台車モデル

制御対象は以下の$x_1$方向にのみ速度$v$を持ち、$y_1$方向には速度を持たない台車とする。<br>
入力$u$は台車の前進方向$x_1$へ加速度を発生させ、入力$\tau$は台車の回転方向$\theta$に角加速度を発生させる。この二つが入力となる。

<img src="images/nonholonomic_car.png" style="width:40%;"/>

台車の前進方向の運動と、回転方向の運動を以下とする。

$$
\begin{aligned}
m \dot{v}(t) = u(t) \\
I \dot{\omega}(t) = \tau(t) \\
\dot{\theta}(t) = \omega(t)
\end{aligned}
$$

$v$を $\Sigma_o$ で表現すると、次のようになる。

$$
\begin{aligned}
\dot{x}(t)= v(t) \cos(\theta(t)) \\
\dot{y}(t) = v(t) \sin(\theta(t))
\end{aligned}
$$

状態 $X$ を次のように設定する。

$$
X = \begin{bmatrix} v(t) & \theta(t) & \omega(t) & x(t) & y(t) \end{bmatrix}^T
$$

よって、状態方程式 $\dot{X} = f(X, U, t)$ は次の式となる。

$$
\dot{X} = \begin{bmatrix}
\dot{v}(t) \\ \dot{\theta}(t) \\ \dot{\omega}(t) \\ \dot{x}(t) \\ \dot{y}(t)
\end{bmatrix} = 
\begin{bmatrix}
u(t)/m \\\omega(t) \\ \tau(t)/I \\ v(t) \cos(\theta(t)) \\ v(t) \sin(\theta(t))
\end{bmatrix}
$$

$$
U=\begin{bmatrix}
u(t) \\ \tau(t)
\end{bmatrix}
$$

#### 非ホロノミック拘束について

台車は $\Sigma_1$ の $y_1$ 方向には横滑りしないという条件がある。

$\Sigma_o$ における台車の速度 $^o v$ を以下のようにベクトルで$x,y$軸それぞれの速度として表現する。

$$
^o v = \begin{bmatrix} \dot{x} \\ \dot{y} \end{bmatrix}
$$

$\Sigma_1$ の$y_1$ 方向の単位ベクトルを$e_{y_1}$とすると、これを$\Sigma_o$ で表すと以下となる。

$$
^o e_{y_1} = \begin{bmatrix} -\sin(\theta) \\ \cos(\theta) \end{bmatrix}
$$

よって、台車の$y_1$方向の速度は内積によって、以下のように表すことが出来る。

$$
v_{y_1} = {^o e_{y_1}}^T \ ^o v = -\dot{x} \sin(\theta) + \dot{y} \cos(\theta)
$$

台車は横滑りしないため、$v_{y_1}=0$であり、これより以下の拘束条件が導かれる。

$$
-\dot{x} \sin(\theta) + \dot{y} \cos(\theta) = 0
$$

これを非ホロノミック拘束と呼ぶ。

拘束条件を以下のように位置、姿勢だけで記述できる場合、これをホロノミック拘束と呼ぶ。

$$
g(x, y, \theta) = 0
$$

今回の拘束条件は以下のように速度に対するものである。

$$
-\dot{x} \sin(\theta) + \dot{y} \cos(\theta) = 0
$$

これを一般的には積分してホロノミック拘束$g(x,y,\theta) = 0$ という位置だけの拘束にはできない。

これは、「横方向に速度を出すことはできないが、横にある位置へ移動することはできる」ということを意味している。

例えば台車の位置が$\Sigma_o$ で $(0,0)$ にあったとする。その後台車は左へ 90deg 旋回、1 前進、右へ90deg 旋回とすれば $(0,1)$ へ到達することが出来る。

よって、位置が拘束されているのではなく、瞬間的に許される速度方向が拘束されているということになる。

車体の前進方向の速度 $v$を$\Sigma_o$で表した方程式を上記の$v_{y_1}$ の式に代入すると以下のようになるため、状態方程式そのものが非ホロノミック拘束を常に満たしている。

$$
v_{y_1} = {^o e_{y_1}}^T \ ^o v = -\dot{x} \sin(\theta) + \dot{y} \cos(\theta) = -v \cos(\theta) \sin(\theta) + v \sin(\theta)\cos(\theta) = 0
$$

## 位置と制御入力の拘束

### 位置拘束

台車はXY平面を動く。そのため、ある範囲の中に入らないように位置の拘束を行う。

この範囲を以下の円の方程式から考える。

$$
(x - x_{c_1})^2 + (y - y_{c_1})^2 = {r_1}^2
$$

この方程式は、$(x_{c_1}, y_{c_1})$を中心に半径 $r_1$ の円である。

以下のように等式の拘束とすると、台車の位置 ($x,y$) は円周上に固定されることになる。

$$
(x - x_{c_1})^2 + (y - y_{c_1})^2 - {r_1}^2 = 0
$$

以下の方程式は、$(x_{c_1}, y_{c_1})$を中心に半径 $r_1$ の円の内側に台車の位置($x,y$)があることにある。これは範囲の外に出ないような拘束に繋がる。

$$
(x - x_{c_1})^2 + (y - y_{c_1})^2 \le {r_1}^2
$$

以下の方程式は、$(x_{c_1}, y_{c_1})$を中心に半径 $r_1$ の円の外側に台車の位置($x,y$)があることにある。これは範囲の中に入らないような拘束に繋がる。

$$
(x - x_{c_1})^2 + (y - y_{c_1})^2 > {r_1}^2
$$

ここから 位置拘束$G_1$を以下のように定義する。

$$
G_1 = (x - x_{c_1})^2 + (y - y_{c_1})^2 - {r_1}^2 > 0
$$

例えば簡単に($x_{c_1},  y_{c_1}$) = ($0,0$)、$r_1 = 1$ として考えると、以下のように円周に正方向から近づくと、$G_1$ は境界の $0$ に近づいていく。

- $(x,y) = (1.1, 0)$ : $G_1 = 1.21 - 1 = 0.21 > 0$
- $(x,y) = (1.001, 0)$ : $G_1 = 1.001^2 - 1 = 0.002001 > 0$

よって、$-\log(G_1)$とすると、境界に近づくにつれ $+\infty$ と壁を高くすることが出来る。これをランニングコストに追加する。

この考え方で、$G_i \ (i = 1, \cdots, m)$ のように複数の位置拘束を考えることが出来る。

今回は位置拘束は2つとする。

$$
G_1 = (x - x_{c_1})^2 + (y - y_{c_1})^2 - {r_1}^2 > 0 \\
G_2 = (x - x_{c_2})^2 + (y - y_{c_2})^2 - {r_2}^2 > 0
$$

計算のため上記を正規化する。

$$
G_1 = \frac{(x - x_{c_1})^2 + (y - y_{c_1})^2}{{r_1}^2} - 1 > 0 \\
G_2 = \frac{(x - x_{c_2})^2 + (y - y_{c_2})^2}{{r_2}^2} - 1 > 0
$$




### 制御入力拘束

入力として $u, \tau$ の二つがある。そこで次のように上限値を設ける。

$$
\begin{split}
\begin{aligned}
u^2 &\le {u_{max}}^2 \\
\tau^2 & \le {\tau_{max}}^2
\end{aligned}
\end{split}
$$

$\log$関数の特徴に合わせるように、次のように 正側の拘束に変換する。

$$
\begin{split}
\begin{aligned}
u^2 - {u_{max}}^2&\le 0  \\
\tau^2 - {\tau_{max}}^2 & \le 0 
\end{aligned}
\end{split} \Rightarrow
\begin{split}
\begin{aligned}
{u_{max}}^2 - u^2 &\ge 0  \\
{\tau_{max}}^2 - \tau^2 & \ge 0 
\end{aligned}
\end{split}
$$

それぞれを以下の拘束とする。

$$
G_u = {u_{max}}^2 - u^2 > 0\\
G_\tau = {\tau_{max}}^2 - \tau^2 > 0
$$

また、こちらも計算のため、以下のように正規化を行う。

$$
G_u = 1 - \left(\frac{u}{u_{max}}\right)^2 > 0\\
G_\tau = 1 - \left(\frac{\tau}{\tau_{max}}\right)^2 > 0
$$


$\log$関数による制御入力拘束は以下となる。

$$
-\log(G_u) , \space -\log(G_\tau)
$$

位置と制御入力の拘束条件をまとめて、以下のように記述する。

$$
G(X,U) = \begin{bmatrix}
G_1 , G_2 , G_u, G_\tau
\end{bmatrix}^T
$$


## 拡大評価関数と各コスト関数


$\log$バリア関数を制約条件に追加した拡大評価関数 $\bar{J}$ は次となる。

$$
\bar{J} = \Phi(X(T)) + \int_0^T \left[ L(X,U,t) - \eta^T \log(G(X,U)) + \lambda^T \left( f(X,U,t) - \dot{X} \right) \right] dt
$$

ここで$\lambda$は状態$X$に合わせて以下のようにする。

$$
\lambda = \begin{bmatrix}
\lambda_v , \lambda_\theta , \lambda_\omega ,\lambda_x, \lambda_y
\end{bmatrix}^T
$$

また、$\eta$は制約に合わせて以下のようにする。今回位置の制約は二つとする。また、この$\eta$は$\log$バリア関数に対応する重みである。

$$
\eta = \begin{bmatrix}
\eta_1 , \eta_2, \eta_u, \eta_\tau
\end{bmatrix}^T
$$

また、表現として$\log(G(X,U))$は以下のようになっているとする。

$$
\log(G(X,U)) = \begin{bmatrix}
\log(G_1) \\
\log(G_2) \\
\log(G_u) \\
\log(G_\tau)
\end{bmatrix}
$$

### 終端コスト $\Phi$

終端の角度目標値との誤差の表現について注意が必要である。

$e = (\theta(T) - \theta_{ref})$とすると、$\theta(T)$が178degから先 3deg に進んで-179degとなった場合、$e$は不連続な値となる。

|e|$\theta(T)$|$\theta_{ref}$|
|---:|---:|---:|
|-1|178|179|
|-358|-179|179|
|-349|-170|179|

そこで、$\cos(e)$とすると、その値を[-1,1]の範囲に納めることが出来る。
最小化問題であるため、$\cos(e) = -1$ が最小値、つまり$e=\pm\pi$と目標とは反対の向きが最小になってしまう。

そこで $1-\cos(e)$とする。すると以下のように、$\theta(T)$が$\theta_{ref}$に近いときはゼロに近く、反転しているときは2 、90deg離れているときは1となり、目標値に近い姿勢を最小とすることができる。

$1-\cos(e)$|e|$\theta(T)$|$\theta_{ref}$|
|---:|---:|---:|---:|
|0.00015|-1|178|179|
|0.00061|-358|-179|179|
|2|-180|-1|179|
|1|-90|89|179|

よって、終端コストは以下のようになる。

$$
\boxed{
\begin{aligned}
\Phi(X(T)) = \frac{1}{2} \Big[ q_{v_T} \ (v(T))^2 +  q_{\omega_T}(\omega(T))^2 \Big] + q_{\theta_T} \ \left(1 - \cos(\theta(T) - \theta_{ref})\right) \\ 
+ \frac{1}{2} \Big[ q_{x_T}(x(T) - x_{ref})^2 + q_{y_T}(y(T) - y_{ref})^2 \Big]
\end{aligned}
}
$$ 

最小化問題に対するコストであるため、それぞれのゲインに関して以下の目的になる。

- $q_{v_T}$ : 最終時刻$T$における速度 $v(T)$ を小さくする
- $q_{\omega_T}$ : 最終時刻$T$における角速度 $\omega(T)$ を小さくする
- $q_{\theta_T}$ : 最終時刻$T$における角度の誤差 $(\theta(T)-\theta_{ref}) $ を小さくする
- $q_{x_T}$ : 最終時刻$T$における$x$位置の誤差 $(x(T)-x_{ref}) $ を小さくする
- $q_{y_T}$ : 最終時刻$T$における$y$位置の誤差 $(y(T)-y_{ref}) $ を小さくする


### ランニングコスト $L$

ランニングコストにおける角度目標値との誤差も終端コストで用いた表現と同じものを用いる。

$$
\boxed{
\begin{aligned}
L(X,U,t) = \frac{1}{2} \left( q_v (v(t))^2 + q_{\omega}(\omega(t))^2 \right) + q_{\theta} \left(1-\cos(\theta(t) - \theta_{ref}) \right) \\
+ \frac{1}{2} \Big(q_x(x(t) - x_{ref})^2 + q_y(y(t) - y_{ref})^2  + q_u (u(t))^2 + q_\tau (\tau(t))^2 \Big) 
\end{aligned}
}
$$

最小化問題に対するコストであるため、それぞれのゲインに関して以下の目的になる。

- $q_{v}$ : 区間$[0,T)$における速度 $v(t)$ を小さくする
- $q_{\omega}$ : 区間$[0,T)$における角速度 $\omega(t)$ を小さくする
- $q_{\theta}$ : 区間$[0,T)$における角度の誤差 $(\theta(t)-\theta_{ref}) $ を小さくする
- $q_{x}$ : 区間$[0,T)$における$x$位置の誤差 $(x(t)-x_{ref}) $ を小さくする
- $q_{y}$ : 区間$[0,T)$における$y位置$の誤差 $(y(t)-y_{ref}) $ を小さくする
- $q_{u}$ : 区間$[0,T)$における入力 $u(t)$ を小さくする
- $q_{\tau}$ : 区間$[0,T)$における入力$\tau(t)$ を小さくする


#### $\log$バリア関数によるコスト $-\eta^T \ \log(G)$の項

$$
\boxed{
\begin{aligned}
-\eta^T \log(G(x,u)) &= -\eta_1 \log \left( \frac{(x(t) - x_{c_1})^2 + (y(t) - y_{c_1})^2}{{r_1}^2} -1 \right) 
- \eta_2 \log \left( \frac{(x(t) - x_{c_2})^2 + (y(t) - y_{c_2})^2}{{r_2}^2} -1 \right) \\
&- \eta_u \log \left( 1 - \left(\frac{u(t)}{{u_{max}}}\right)^2 \right) 
- \eta_\tau \log \left( 1 - \left(\frac{\tau(t)}{{\tau_{max}}} \right)^2 \right) \\
\end{aligned}
}
$$

$\eta$はゲインとして取り扱う。そのため、$\eta=0$とすると、この拘束自体を評価から除外することが可能になっている。

評価関数では $-\eta^T \log(G(x,u))$ で評価を行うため、境界から離れるほどコストは小さくなっていく。
また、境界に近づくほど $+\infty$ となるため、境界へ近づかないような挙動になる。

### Hamiltonianの構成

$H$ は以下のように $\log$バリア関数を追加した形になる。

$$
H = L - \eta^T \log(G) + f^T \lambda
$$

$f^T \lambda$の項は以下である。

$$
\boxed{
f^T \lambda = \lambda_v u(t)/m + \lambda_\theta \omega(t) + \lambda_\omega \tau(t)/I + \lambda_x v(t) \cos(\theta(t)) + \lambda_y v(t) \sin(\theta(t))
}
$$

これを$X, U, \lambda$ で偏微分を行う。

#### $X$による偏微分

$$
X = \begin{bmatrix} v(t) & \theta(t) & \omega(t) & x(t) & y(t) \end{bmatrix}^T
$$


$$
H_X = \frac{\partial H}{\partial X} =
\begin{bmatrix}
{\partial H}/{\partial v} \\ {\partial H}/{\partial \theta} \\ {\partial H}/{\partial \omega} \\ {\partial H}/{\partial x} \\ {\partial H}/{\partial y}
\end{bmatrix} \\
$$

$$
\boxed{
\begin{array}{l}
\dfrac{\partial H}{\partial v} = q_v v(t) + \lambda_x \cos(\theta(t)) + \lambda_y \sin(\theta(t)) \\[8pt]
\dfrac{\partial H}{\partial \theta} = q_\theta (\sin(\theta(t) - \theta_{ref}) ) - \lambda_x v(t)\sin(\theta(t)) + \lambda_y v(t) \cos(\theta(t)) \\[8pt]
\dfrac{\partial H}{\partial \omega} = q_\omega \omega(t) + \lambda_\theta \\[8pt] 
\begin{aligned}
\frac{\partial H}{\partial x} =
q_x(x(t) - x_{ref}) - \sum_{i=1}^2 \eta_i \dfrac{2(x(t) - x_{c_i})}{r_i^2} \left({\dfrac{(x(t)-x_{c_i})^2 + (y(t)-y_{c_i})^2}{{r_i}^2} -1}\right)^{-1}
\end{aligned} \\
\begin{aligned}
\frac{\partial H}{\partial y} =
q_y(y(t) - y_{ref}) - \sum_{i=1}^2 \eta_i \dfrac{2(y(t)-y_{c_i})}{r_i^2} \left({\dfrac{(x(t)-x_{c_i})^2 + (y(t)-y_{c_i})^2}{{r_i}^2} -1}\right)^{-1} 
\end{aligned}
\end{array}
}
$$

#### $U$ による偏微分

$$
U = \begin{bmatrix} u(t) & \tau(t) \end{bmatrix}^T
$$

$$
H_U = \frac{\partial H}{\partial U} =\begin{bmatrix} \partial H / \partial u \\ \partial H / \partial \tau \end{bmatrix}
$$

$$
\boxed{
\begin{array}{l}
\dfrac{\partial H}{\partial u} = q_u u(t) + 2\eta_u \left( \dfrac{u(t)}{{u_{max}}^2}\right)\left( 1 - \left( \dfrac{u(t)}{u_{max}}\right)^2\right)^{-1} + \dfrac{\lambda_v}{m} \\
\dfrac{\partial H}{\partial \tau} = q_\tau \tau(t) + 2\eta_\tau \left( \dfrac{\tau(t)}{{\tau_{max}}^2}\right)\left( 1 - \left( \dfrac{\tau(t)}{\tau_{max}}\right)^2\right)^{-1} + \dfrac{\lambda_\omega}{I} \\
\end{array}
}
$$

### $\lambda$ による偏微分

$H_\lambda = \dfrac{\partial H}{\partial \lambda} = f(X,U,t)$より、

$$
\boxed{
H_\lambda = \begin{bmatrix}
u(t)/m \\\omega(t) \\ \tau(t)/I \\ v(t) \cos(\theta(t)) \\ v(t) \sin(\theta(t))
\end{bmatrix}
}
$$

#### 終端条件 $\Phi_X$

$$
\Phi_X(X(T)) = \frac{\partial \Phi(X(T))}{\partial X(T)} =
\begin{bmatrix}
\partial \Phi(X(T)) / \partial v(T) \\
\partial \Phi(X(T)) / \partial \theta(T) \\
\partial \Phi(X(T)) / \partial \omega(T) \\
\partial \Phi(X(T)) / \partial x(T) \\
\partial \Phi(X(T)) / \partial y(T) \\
\end{bmatrix}
$$

$$
\boxed{
\begin{array}{l}
\dfrac{\partial \Phi(X(T))}{\partial v(T)} = q_{v_T} v(T) \\[8pt]
\dfrac{\partial \Phi(X(T))}{\partial \theta(T)} = q_{\theta_T}\sin(\theta(T) - \theta_{ref}) \\[8pt]
\dfrac{\partial \Phi(X(T))}{\partial \omega(T)} = q_{\omega_T} \omega(T) \\[8pt]
\dfrac{\partial \Phi(X(T))}{\partial x(T)} = q_{x_T} (x(T) -x_{ref}) \\[8pt]
\dfrac{\partial \Phi(X(T))}{\partial y(T)} = q_{y_T} (y(T) - y_{ref}) \\[8pt]
\end{array}
}
$$


## PMPの必要条件式

状態方程式、終端条件、随伴方程式、停留条件は以下となる。

$$
\boxed{
\begin{aligned}
\dot{X} &= H_\lambda \\
\lambda(T) &= \Phi_X(X(T)) \\
\dot{\lambda} &= - H_X \\
H_U &= 0
\end{aligned}
}
$$

## C/GMRESでの計算



### 停留条件式 $F$の計算

C/GMRESでは予測ホライゾンの離散時間$\tau(tで状態$X$と随伴変数$

### 初期 $U(0)$ の計算

### C/GMRES 計算ステップ